# Fine-tune DeBERTa-v3 and RoBERTa on silver labels (Component 1)

Token classification with BIO tags for five fields: deadline, fee, career_stage, location, medium.
Train on silver train, tune and stop early on silver val. Gold is NOT used here.

Three runs are trained one after the other in one run (DeBERTa, RoBERTa, and RoBERTa with more label-free windows), with the same data, windows, settings, seed, and scorer:
`deberta` = microsoft/deberta-v3-base and `roberta` = roberta-base. Each model has its own token files because
the two tokenizers cut text differently. A model that already has a results file in Drive is skipped, so if the
session disconnects you can rerun the cells and it continues with the model that is left.

**Before running:** in Google Drive, put the files like this:
- in `DLArtManager/data/` (7 data files):
- from `data/labeled/`: `tokens_train.jsonl`, `tokens_val.jsonl`, `label_map.json` (DeBERTa),
  `tokens_roberta_train.jsonl`, `tokens_roberta_val.jsonl`, `label_map_roberta.json` (RoBERTa), `eval_val.json`
- in `DLArtManager/` itself (2 code files, from `models/extraction/`): `decode.py`, `evaluate.py`

Set the runtime to **GPU** (Runtime > Change runtime type > T4 GPU) and run the cells in order.

In [ ]:
!pip -q install seqeval accelerate
import os, sys, json, random, time, gc, importlib.util
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/DLArtManager'
LOCAL = '/content/work'   # models and checkpoints stay on Colab's disk (Drive is nearly full); small result files go to Drive
os.makedirs(LOCAL, exist_ok=True)
DATA = f'{DRIVE}/data'                 # all input files live here; results and models are written to DRIVE
# name: (which token files to use, label-free windows per labeled window in training)
EXPERIMENTS = {'deberta': ('deberta', 1), 'roberta': ('roberta', 1), 'roberta_neg3': ('roberta', 3)}
RUNS = list(EXPERIMENTS)               # trained in this order; a run with a results file in Drive is skipped
FP16 = True                            # set False if the loss becomes nan
FILES = {   # the token files are specific to each model's tokenizer
    'deberta': {'train': 'tokens_train.jsonl', 'val': 'tokens_val.jsonl', 'labels': 'label_map.json'},
    'roberta': {'train': 'tokens_roberta_train.jsonl', 'val': 'tokens_roberta_val.jsonl', 'labels': 'label_map_roberta.json'},
}

assert os.path.isdir(DRIVE), f'Folder not found: {DRIVE}. It must be directly inside My Drive.'
assert os.path.isdir(DATA), f'Folder not found: {DATA}. Create a folder named data inside DLArtManager.'
need_data = ['eval_val.json'] + [f for r in sorted({b for b, _ in EXPERIMENTS.values()}) for f in FILES[r].values()]
missing = [f for f in need_data if not os.path.exists(f'{DATA}/{f}')]
missing += [f for f in ['decode.py', 'evaluate.py'] if not os.path.exists(f'{DRIVE}/{f}')]   # code files sit in DLArtManager, not in data
assert not missing, f'Missing: {missing}  (data files go in {DATA}, decode.py and evaluate.py go in {DRIVE})'
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the runtime type')

# load our own scoring and decoding files by path (a Hugging Face package is also called "evaluate")
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

ev = load_module('aof_evaluate', f'{DRIVE}/evaluate.py')
dec = load_module('aof_decode', f'{DRIVE}/decode.py')
ev_val = json.load(open(f'{DATA}/eval_val.json'))
gold = {lid: [tuple(s) for s in v['spans']] for lid, v in ev_val.items()}
texts = {lid: v['text'] for lid, v in ev_val.items()}

def read(p):
    return [json.loads(l) for l in open(p)]

In [ ]:
import numpy as np
from datasets import Dataset
from torch.nn.functional import softmax
from seqeval.metrics import f1_score, precision_score, recall_score
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification, EarlyStoppingCallback, set_seed)

def to_ds(ws):
    return Dataset.from_list([{'input_ids': w['input_ids'], 'attention_mask': [1] * len(w['input_ids']),
                               'labels': w['labels']} for w in ws])

def run_model(RUN):
    """Train one model on silver train, stop early on silver val, score val, save everything to Drive."""
    base, neg_ratio = EXPERIMENTS[RUN]
    files = FILES[base]
    lm = json.load(open(f'{DATA}/{files["labels"]}'))
    LABELS, label2id = lm['labels'], lm['label2id']
    id2label = {i: l for i, l in enumerate(LABELS)}
    MODEL = lm['model']
    train_w, val_w = read(f'{DATA}/{files["train"]}'), read(f'{DATA}/{files["val"]}')
    print(f'=== {RUN}: {MODEL} | {len(train_w)} train windows, {len(val_w)} val windows, {len(LABELS)} labels')

    # Most windows have no label at all. Keep every window that has a label and neg_ratio times as many
    # label-free windows (1 = equal numbers). More label-free windows teach the model to predict fewer spans.
    random.seed(42)
    pos = [w for w in train_w if any(l > 0 for l in w['labels'])]
    neg = [w for w in train_w if not any(l > 0 for l in w['labels'])]
    train_used = pos + random.sample(neg, min(len(neg), int(neg_ratio * len(pos))))
    random.shuffle(train_used)
    print(f'train windows: {len(train_w)} -> {len(train_used)} ({len(pos)} with labels, neg_ratio={neg_ratio})')
    train_ds, val_ds = to_ds(train_used), to_ds(val_w)   # val uses ALL windows, unsampled

    set_seed(42)
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    # Newer transformers load the weights in the checkpoint's own precision (fp16 here). Mixed precision
    # training (fp16=True) needs float32 weights, or it fails with 'Attempting to unscale FP16 gradients'.
    model = model.float()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'{n_params / 1e6:.0f}M parameters')

    def compute_metrics(p):
        preds = p.predictions.argmax(-1)
        true_l, pred_l = [], []
        for pr, lb in zip(preds, p.label_ids):
            keep = lb != -100
            true_l.append([id2label[int(x)] for x in lb[keep]])
            pred_l.append([id2label[int(x)] for x in pr[keep]])
        return {'precision': precision_score(true_l, pred_l), 'recall': recall_score(true_l, pred_l), 'f1': f1_score(true_l, pred_l)}

    # identical settings for both models, so the comparison is fair
    EPOCHS, BATCH = 8, 8
    # 10% of all training steps for warmup. Newer transformers versions removed warmup_ratio,
    # so the step count is computed here and works in old and new versions.
    warmup_steps = int(0.1 * EPOCHS * -(-len(train_ds) // BATCH))
    args = TrainingArguments(
        output_dir=f'{LOCAL}/checkpoints/{RUN}', learning_rate=2e-5, per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=16, num_train_epochs=EPOCHS, weight_decay=0.01, warmup_steps=warmup_steps,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1',
        greater_is_better=True, save_total_limit=2, fp16=FP16, seed=42, report_to='none', logging_steps=50)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorForTokenClassification(tok), compute_metrics=compute_metrics,
                      processing_class=tok, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    t0 = time.time()
    trainer.train()
    train_seconds = time.time() - t0
    trainer.save_model(f'{LOCAL}/model_{RUN}')
    tok.save_pretrained(f'{LOCAL}/model_{RUN}')
    print('best val token-level F1 (seqeval):', trainer.state.best_metric, '| training time (s):', round(train_seconds))

    # Score on silver val the same way the regex baseline is scored: page-level spans through evaluate.py.
    model.eval()
    t0 = time.time()
    probs = []
    for i in range(0, len(val_w), 16):
        batch = val_w[i:i + 16]
        L = max(len(w['input_ids']) for w in batch)
        ids = torch.tensor([w['input_ids'] + [tok.pad_token_id] * (L - len(w['input_ids'])) for w in batch]).to(model.device)
        mask = torch.tensor([[1] * len(w['input_ids']) + [0] * (L - len(w['input_ids'])) for w in batch]).to(model.device)
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask).logits
        pr = softmax(out.float(), -1).cpu().numpy()
        for w, p in zip(batch, pr):
            probs.append(p[:len(w['input_ids'])])
    infer_seconds = time.time() - t0

    by_page = dec.decode_all(val_w, probs, id2label)
    pred = {lid: by_page.get(v['page_key'], []) for lid, v in ev_val.items()}
    res = ev.score(gold, pred, texts)
    res['run'] = {'name': RUN, 'model': MODEL, 'params_millions': round(n_params / 1e6),
                  'train_seconds': round(train_seconds), 'val_inference_seconds': round(infer_seconds, 1),
                  'val_windows': len(val_w), 'best_epoch_f1_seqeval': trainer.state.best_metric}
    report = ev.to_markdown(res, f'{MODEL} ({RUN}) on silver val (check only, not the final number)')
    print(report)
    json.dump(res, open(f'{DRIVE}/results_val_{RUN}.json', 'w'), indent=1)
    open(f'{DRIVE}/results_val_{RUN}.md', 'w').write(report)
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()   # free the GPU before the next model
    return res

In [ ]:
# Train each model in turn. A model with a results file in Drive is skipped.
for RUN in RUNS:
    if os.path.exists(f'{DRIVE}/results_val_{RUN}.json'):
        print(f'{RUN}: results already in Drive, skipping (delete results_val_{RUN}.json to retrain)')
        continue
    run_model(RUN)

In [ ]:
# For each saved model: save its predictions on silver val, and try making the decoder more cautious.
# o_scale multiplies the "no label" probability: 1.0 is the plain result, bigger means fewer spans.
# This uses the saved models, no training. Picking o_scale on val is tuning on val, so the best value is
# chosen here and then fixed before gold is scored.
def val_probs(RUN):
    base, _ = EXPERIMENTS[RUN]
    files = FILES[base]
    lm = json.load(open(f'{DATA}/{files["labels"]}'))
    id2label = {i: l for i, l in enumerate(lm['labels'])}
    val_w = read(f'{DATA}/{files["val"]}')
    tok = AutoTokenizer.from_pretrained(f'{LOCAL}/model_{RUN}')
    model = AutoModelForTokenClassification.from_pretrained(f'{LOCAL}/model_{RUN}').float().to('cuda').eval()
    probs = []
    for i in range(0, len(val_w), 16):
        batch = val_w[i:i + 16]
        L = max(len(w['input_ids']) for w in batch)
        ids = torch.tensor([w['input_ids'] + [tok.pad_token_id] * (L - len(w['input_ids'])) for w in batch]).to('cuda')
        mask = torch.tensor([[1] * len(w['input_ids']) + [0] * (L - len(w['input_ids'])) for w in batch]).to('cuda')
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask).logits
        pr = softmax(out.float(), -1).cpu().numpy()
        for w, p in zip(batch, pr):
            probs.append(p[:len(w['input_ids'])])
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return val_w, probs, id2label

SCALES = (1.0, 1.5, 2.0, 3.0, 5.0)
for RUN in RUNS:
    val_w, probs, id2label = val_probs(RUN)
    print(f'\n=== {RUN}: exact span P / R / F1 by o_scale')
    print('| o_scale | micro P / R / F1 | deadline F1 | fee F1 | career F1 | location F1 | medium F1 | spans |')
    print('|---|---|---|---|---|---|---|---|')
    sweep = {}
    for s in SCALES:
        by_page = dec.decode_all(val_w, probs, id2label, o_scale=s)
        pred = {lid: [tuple(x) for x in by_page.get(v['page_key'], [])] for lid, v in ev_val.items()}
        r = ev.score(gold, pred, texts)
        e = r['span']['exact']
        m = e['micro']
        print(f"| {s} | {m['precision']:.2f} / {m['recall']:.2f} / {m['f1']:.2f} | "
              + ' | '.join(f"{e[f]['f1']:.2f}" for f in ev.FIELDS) + f" | {m['n_pred']} |")
        sweep[str(s)] = r
        if s == 1.0:
            json.dump({k: [list(x) for x in v] for k, v in pred.items()}, open(f'{DRIVE}/predictions_val_{RUN}.json', 'w'))
    json.dump(sweep, open(f'{DRIVE}/sweep_val_{RUN}.json', 'w'))

In [ ]:
# All runs that have results, side by side on silver val (exact span P / R / F1).
done = {r: json.load(open(f'{DRIVE}/results_val_{r}.json')) for r in RUNS if os.path.exists(f'{DRIVE}/results_val_{r}.json')}
print('results found for:', list(done))
names = list(done)
rows = ['| Field | ' + ' | '.join(names) + ' |', '|---|' + '---|' * len(names)]
for f in ev.FIELDS + ['micro']:
    cells = [done[n]['span']['exact'][f] for n in names]
    rows.append(f'| {f} | ' + ' | '.join(f"{c['precision']:.2f} / {c['recall']:.2f} / {c['f1']:.2f}" for c in cells) + ' |')
for name in ('deadline', 'fee'):
    cells = [done[n]['value'][name] for n in names]
    rows.append(f'| {name} (value match) | ' + ' | '.join(f"{c['precision']:.2f} / {c['recall']:.2f} / {c['f1']:.2f}" for c in cells) + ' |')
rows += ['', '| | ' + ' | '.join(names) + ' |', '|---|' + '---|' * len(names)]
for k, label in (('params_millions', 'Parameters (millions)'), ('train_seconds', 'Training time (s)'),
                 ('val_inference_seconds', 'Val inference (s)')):
    rows.append(f'| {label} | ' + ' | '.join(str(done[n]['run'][k]) for n in names) + ' |')
rows.append('| Predicted spans (gold 244) | ' + ' | '.join(str(done[n]['span']['exact']['micro']['n_pred']) for n in names) + ' |')
table = '\n'.join(rows)
print(table)
open(f'{DRIVE}/comparison_val.md', 'w').write(table)

**Send back:** the per-epoch table from each model and the final comparison table.
Scores are against silver val, a development check, with only 32 listings and a single seed per model.
A difference of a few points between the two models is not reliable. The number to report comes from gold,
scored once at the end. Compare with `docs/results/regex_baseline_silver_val_check.md`.